# Credit Card Fraud Detection Agent

Cleaned notebook — final agent pipeline only. Exploratory/debugging cells have been removed and the execution order has been fixed.

In [20]:
# 1. Imports and data loading

import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df_trans = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\week1\fraud_transactions.csv")
df_cust = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\customer_profiles.csv")

df_trans["timestamp"] = pd.to_datetime(df_trans["timestamp"])

print("Transactions:", df_trans.shape)
print("Customers:", df_cust.shape)

Transactions: (17000, 8)
Customers: (1000, 5)


In [21]:
# 2. Add customer profile information

df_trans = df_trans.merge(df_cust[["customer_id", "home_location"]],on="customer_id",how="left")

print(df_trans[["customer_id", "location", "home_location"]].head())

  customer_id   location home_location
0       C0650       Pune          Pune
1       C0751        NaN     Bengaluru
2       C0187    Chennai       Chennai
3       C0330  Hyderabad     Hyderabad
4       C0500  Hyderabad     Hyderabad


In [22]:
# 3. Build customer histories

customer_histories = {}

for customer_id, group in df_trans.groupby("customer_id"):
    customer_histories[customer_id] = group.sort_values("timestamp").copy()

print("Customer histories:", len(customer_histories))

Customer histories: 1000


In [23]:
# 4. Evidence functions

def get_customer_history(customer_id, timestamp, customer_histories):
    customer = customer_histories[customer_id]
    current_time = pd.to_datetime(timestamp)

    history = customer.loc[customer["timestamp"] < current_time]

    return history.drop(columns="fraud")


def check_amount(current_amount, history):
    if history.empty:
        return False

    avg_amount = history["amount"].mean()

    if current_amount > 4 * avg_amount:
        return True

    return False


def check_location(current_location, historical_locations):
    if pd.isna(current_location):
        return None

    if current_location in historical_locations.values:
        return False

    return True


def check_merchant(current_merchant, historical_merchants):
    if pd.isna(current_merchant):
        return None

    if current_merchant in historical_merchants.values:
        return False

    return True


def check_frequency(current_timestamp, history):
    if history.empty:
        return False

    current_timestamp = pd.to_datetime(current_timestamp)

    # Look at the previous 24 hours
    one_day_ago = current_timestamp - pd.Timedelta(hours=24)

    recent_transactions = history[
        (history["timestamp"] >= one_day_ago) &
        (history["timestamp"] < current_timestamp)
    ]

    normal_daily_frequency = (
        history["timestamp"].dt.date.value_counts().median()
    )

    return len(recent_transactions) >= 2 * normal_daily_frequency


def check_velocity(timestamp, history):
    if history.empty:
        return False

    recent_transactions = history[
        ((timestamp - history["timestamp"]).dt.total_seconds() / 60) <= 30
    ]

    return len(recent_transactions) >= 3

In [24]:
# 5. Generate evidence for every transaction

def get_evidence(transaction, customer_histories):
    history = get_customer_history(transaction["customer_id"],transaction["timestamp"],customer_histories)

    amount_unusual = check_amount(transaction["amount"],history)

    location_unusual = check_location(transaction["location"],history["location"])

    merchant_unusual = check_merchant(transaction["merchant"],history["merchant"])

    frequency_unusual = check_frequency(transaction["timestamp"],history)

    velocity_unusual = check_velocity(transaction["timestamp"],history)

    return {
        "amount_unusual": amount_unusual,
        "location_unusual": location_unusual,
        "merchant_unusual": merchant_unusual,
        "frequency_unusual": frequency_unusual,
        "velocity_unusual": velocity_unusual
    }


evidence_rows = []

for _, row in df_trans.iterrows():
    evidence_rows.append(
        get_evidence(row, customer_histories)
    )

evidence_df = pd.DataFrame(evidence_rows, index=df_trans.index)

for column in evidence_df.columns:
    df_trans[column] = evidence_df[column]

print(df_trans[
    [
        "amount_unusual",
        "location_unusual",
        "merchant_unusual",
        "frequency_unusual",
        "velocity_unusual"
    ]
].value_counts(dropna=False))

amount_unusual  location_unusual  merchant_unusual  frequency_unusual  velocity_unusual
False           False             False             False              False               7772
                                                    True               True                2771
                                                                       False               1383
                True              True              False              False                957
                False             True              False              False                950
                True              False             True               False                921
True            False             False             True               False                918
False           NaN               False             False              False                373
                False             NaN               False              False                296
                NaN               False         

In [25]:
# 6. Calculate likelihoods

def calculate_likelihoods(df_trans):
    fraud_trans = df_trans.loc[df_trans["fraud"] == 1]
    legit_trans = df_trans.loc[df_trans["fraud"] == 0]

    total_fraud = len(fraud_trans)
    total_legit = len(legit_trans)

    fraud_amount_count = (fraud_trans["amount_unusual"] == True).sum()
    legit_amount_count = (legit_trans["amount_unusual"] == True).sum()

    fraud_location_count = (fraud_trans["location_unusual"] == True).sum()
    legit_location_count = (legit_trans["location_unusual"] == True).sum()

    fraud_merchant_count = (fraud_trans["merchant_unusual"] == True).sum()
    legit_merchant_count = (legit_trans["merchant_unusual"] == True).sum()

    fraud_frequency_count = (fraud_trans["frequency_unusual"] == True).sum()
    legit_frequency_count = (legit_trans["frequency_unusual"] == True).sum()

    fraud_velocity_count = (fraud_trans["velocity_unusual"] == True).sum()
    legit_velocity_count = (legit_trans["velocity_unusual"] == True).sum()

    # Laplace smoothing
    p_amount_fraud = (fraud_amount_count + 1) / (total_fraud + 2)
    p_amount_legit = (legit_amount_count + 1) / (total_legit + 2)

    p_location_fraud = (fraud_location_count + 1) / (total_fraud + 2)
    p_location_legit = (legit_location_count + 1) / (total_legit + 2)

    p_merchant_fraud = (fraud_merchant_count + 1) / (total_fraud + 2)
    p_merchant_legit = (legit_merchant_count + 1) / (total_legit + 2)

    p_frequency_fraud = (fraud_frequency_count + 1) / (total_fraud + 2)
    p_frequency_legit = (legit_frequency_count + 1) / (total_legit + 2)

    p_velocity_fraud = (fraud_velocity_count + 1) / (total_fraud + 2)
    p_velocity_legit = (legit_velocity_count + 1) / (total_legit + 2)

    return (
        p_amount_fraud, p_amount_legit,
        p_location_fraud, p_location_legit,
        p_merchant_fraud, p_merchant_legit,
        p_frequency_fraud, p_frequency_legit,
        p_velocity_fraud, p_velocity_legit
    )


likelihoods = calculate_likelihoods(df_trans)
print(likelihoods)

(np.float64(0.14295915452727792), np.float64(9.998000399920016e-05), np.float64(0.13667523564695802), np.float64(0.10007998400319935), np.float64(0.0005712653527563553), np.float64(0.19976004799040192), np.float64(0.9294487289345901), np.float64(9.998000399920016e-05), np.float64(0.4285918309054556), np.float64(9.998000399920016e-05))


In [26]:
# 7. Calculate fraud belief

def calculate_fraud_belief(evidence, likelihoods):

    prior_fraud = 0.05
    prior_legit = 1 - prior_fraud

    (
        p_amount_fraud,
        p_amount_legit,
        p_location_fraud,
        p_location_legit,
        p_merchant_fraud,
        p_merchant_legit,
        p_frequency_fraud,
        p_frequency_legit,
        p_velocity_fraud,
        p_velocity_legit
    ) = likelihoods

    if evidence["amount_unusual"] is True:
        amount_fraud = p_amount_fraud
        amount_legit = p_amount_legit
    else:
        amount_fraud = 1 - p_amount_fraud
        amount_legit = 1 - p_amount_legit

    if evidence["location_unusual"] is True:
        location_fraud = p_location_fraud
        location_legit = p_location_legit
    elif evidence["location_unusual"] is False:
        location_fraud = 1 - p_location_fraud
        location_legit = 1 - p_location_legit
    else:
        location_fraud = 1
        location_legit = 1

    if evidence["merchant_unusual"] is True:
        merchant_fraud = p_merchant_fraud
        merchant_legit = p_merchant_legit
    elif evidence["merchant_unusual"] is False:
        merchant_fraud = 1 - p_merchant_fraud
        merchant_legit = 1 - p_merchant_legit
    else:
        merchant_fraud = 1
        merchant_legit = 1

    if evidence["frequency_unusual"] is True:
        frequency_fraud = p_frequency_fraud
        frequency_legit = p_frequency_legit
    else:
        frequency_fraud = 1 - p_frequency_fraud
        frequency_legit = 1 - p_frequency_legit

    if evidence["velocity_unusual"] is True:
        velocity_fraud = p_velocity_fraud
        velocity_legit = p_velocity_legit
    else:
        velocity_fraud = 1 - p_velocity_fraud
        velocity_legit = 1 - p_velocity_legit

    fraud_probability = (prior_fraud * amount_fraud * location_fraud * merchant_fraud * frequency_fraud * velocity_fraud)

    legit_probability = (prior_legit  * amount_legit * location_legit * merchant_legit * frequency_legit * velocity_legit)

    return fraud_probability / (fraud_probability + legit_probability)

In [27]:
# 8. Cost model and action selection

def calculate_action_costs(fraud_belief):
    legit_belief = 1 - fraud_belief

    approve_cost = fraud_belief * 100
    question_cost = (fraud_belief * 10 + legit_belief * 5)
    examine_cost = (fraud_belief * 5  + legit_belief * 8)
    decline_cost = legit_belief * 20

    return {
        "Approve": approve_cost,
        "Question": question_cost,
        "Examine": examine_cost,
        "Decline": decline_cost
    }


def choose_action(costs):
    return min(costs, key=costs.get)

In [28]:
# 9. Missing-information handling

def get_questions(evidence):
    questions = []

    if evidence["location_unusual"] is None:
        questions.append("location")

    if evidence["merchant_unusual"] is None:
        questions.append("merchant")

    return questions


def update_transaction(transaction, answers):
    transaction = transaction.copy()

    for question, answer in answers.items():
        if answer is not None:
            transaction[question] = answer

    return transaction


def simulate_customer_response(transaction, questions):
    answers = {}

    for question in questions:
        if question == "location":
            answers["location"] = transaction["home_location"]

        elif question == "merchant":
            # Keep None to simulate an unanswered merchant question.
            answers["merchant"] = None

    return answers

In [29]:
# 10. Final transaction-processing function

def process_transaction(transaction, likelihoods, customer_histories):

    # Step 1: Observe transaction
    evidence = get_evidence(transaction,customer_histories )

    # Step 2: Identify missing information
    questions = get_questions(evidence)

    # Step 3: Ask for missing information
    if questions:
        answers = simulate_customer_response(transaction,questions)

        transaction = update_transaction(transaction,answers)

    # Step 4: Recalculate evidence after questioning
    evidence = get_evidence(transaction,customer_histories )

    # Step 5: Calculate fraud belief
    fraud_belief = calculate_fraud_belief(evidence,likelihoods)

    # Step 6: Calculate expected costs
    costs = calculate_action_costs(fraud_belief)

    # Step 7: Choose lowest-cost action
    action = choose_action(costs)

    return {
        "evidence": evidence,
        "fraud_belief": fraud_belief,
        "costs": costs,
        "questions": questions,
        "action": action
    }

In [30]:
# 11. Test the agent on one transaction

test_transaction = df_trans.iloc[0].copy()

result = process_transaction(test_transaction,likelihoods,customer_histories)

print(result)

{'evidence': {'amount_unusual': False, 'location_unusual': True, 'merchant_unusual': True, 'frequency_unusual': False, 'velocity_unusual': False}, 'fraud_belief': np.float64(7.10392300467347e-06), 'costs': {'Approve': np.float64(0.000710392300467347), 'Question': np.float64(5.0000355196150235), 'Examine': np.float64(7.9999786882309865), 'Decline': np.float64(19.999857921539906)}, 'questions': [], 'action': 'Approve'}


In [31]:
# 12. Run the agent on the complete dataset

actions = []
beliefs = []

for _, row in df_trans.iterrows():

    result = process_transaction(
        row,
        likelihoods,
        customer_histories
    )

    actions.append(result["action"])
    beliefs.append(result["fraud_belief"])

df_trans["agent_action"] = actions
df_trans["fraud_belief"] = beliefs

print(df_trans["agent_action"].value_counts())

agent_action
Approve    10493
Decline     6506
Examine        1
Name: count, dtype: int64


In [32]:
# 13. Evaluate decisions

print("Confusion matrix:")
print(
    pd.crosstab(
        df_trans["fraud"],
        df_trans["agent_action"]
    )
)

y_true = df_trans["fraud"]
y_pred = (
    df_trans["agent_action"] == "Decline"
).astype(int)

print("\nClassification metrics:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))

Confusion matrix:
agent_action  Approve  Decline  Examine
fraud                                  
0               10000        0        0
1                 493     6506        1

Classification metrics:
Accuracy: 0.9709411764705882
Precision: 1.0
Recall: 0.9294285714285714
F1 Score: 0.9634236635569376


In [33]:
# 14. Calculate actual cost

def calculate_actual_cost(row):

    if row["agent_action"] == "Approve":
        return 100 if row["fraud"] == 1 else 0

    if row["agent_action"] == "Question":
        return 10 if row["fraud"] == 1 else 5

    if row["agent_action"] == "Examine":
        return 5 if row["fraud"] == 1 else 8

    if row["agent_action"] == "Decline":
        return 0 if row["fraud"] == 1 else 20


df_trans["actual_cost"] = df_trans.apply(
    calculate_actual_cost,
    axis=1
)

print("Total cost:", df_trans["actual_cost"].sum())
print("Average cost:", df_trans["actual_cost"].mean())

print(
    df_trans.groupby("agent_action")["actual_cost"].agg(
        ["count", "sum", "mean"]
    )
)

Total cost: 49305
Average cost: 2.900294117647059
              count    sum     mean
agent_action                       
Approve       10493  49300  4.69837
Decline        6506      0  0.00000
Examine           1      5  5.00000


In [34]:
# 15. Final diagnostics

print("Fraud belief summary:")
print(df_trans["fraud_belief"].describe())

print("\nAction distribution:")
print(df_trans["agent_action"].value_counts())

print("\nEvidence distribution:")
print(
    df_trans[
        [
            "amount_unusual",
            "location_unusual",
            "merchant_unusual",
            "frequency_unusual",
            "velocity_unusual"
        ]
    ].value_counts(dropna=False)
)

Fraud belief summary:
count    17000.000000
mean         0.383353
std          0.484622
min          0.000005
25%          0.002175
50%          0.002175
75%          0.997560
max          1.000000
Name: fraud_belief, dtype: float64

Action distribution:
agent_action
Approve    10493
Decline     6506
Examine        1
Name: count, dtype: int64

Evidence distribution:
amount_unusual  location_unusual  merchant_unusual  frequency_unusual  velocity_unusual
False           False             False             False              False               7772
                                                    True               True                2771
                                                                       False               1383
                True              True              False              False                957
                False             True              False              False                950
                True              False             True       

In [35]:
false_approvals = df_trans[
    (df_trans["fraud"] == 1) &
    (df_trans["agent_action"] == "Approve")
]

print(false_approvals[
    [
        "transaction_id",
        "amount",
        "location",
        "merchant",
        "amount_unusual",
        "location_unusual",
        "merchant_unusual",
        "frequency_unusual",
        "velocity_unusual",
        "fraud_belief",
        "agent_action"
    ]
].head(20))

      transaction_id   amount   location   merchant  amount_unusual  \
9074    FREQ_C0495_1  3536.38       Pune       Zara           False   
9228    FREQ_C0792_1  1588.15     Mumbai   Reliance           False   
9239    FREQ_C0638_1  3511.73      Delhi   Reliance           False   
9267    FREQ_C0561_1  3849.77    Kolkata     Amazon           False   
9388    FREQ_C0794_1  3261.07    Kolkata        Max           False   
9421    FREQ_C0324_1  3894.76      Delhi        Max           False   
9443    FREQ_C0112_1  3360.97       Pune  Big Chill           False   
9525    FREQ_C0108_1   913.44       Pune   Flipkart           False   
9555    FREQ_C0378_1  3253.14  Hyderabad  Big Chill           False   
9688    FREQ_C0882_1  4524.90  Hyderabad     Amazon           False   
9697    FREQ_C0603_1  2562.85       Pune        H&M           False   
9750    FREQ_C0079_1  1367.98  Hyderabad     Amazon           False   
9768    FREQ_C0044_1  4197.40    Kolkata     Amazon           False   
9854  

In [36]:
print(false_approvals.shape)

(493, 17)


In [37]:
print(
    false_approvals[
        [
            "transaction_id",
            "customer_id",
            "amount",
            "location",
            "merchant",
            "amount_unusual",
            "location_unusual",
            "merchant_unusual",
            "frequency_unusual",
            "velocity_unusual",
            "fraud_belief",
            "agent_action"
        ]
    ].head(5).to_string(index=False)
)

transaction_id customer_id  amount location merchant  amount_unusual location_unusual merchant_unusual  frequency_unusual  velocity_unusual  fraud_belief agent_action
  FREQ_C0495_1       C0495 3536.38     Pune     Zara           False            False            False              False             False      0.002175      Approve
  FREQ_C0792_1       C0792 1588.15   Mumbai Reliance           False            False            False              False             False      0.002175      Approve
  FREQ_C0638_1       C0638 3511.73    Delhi Reliance           False            False            False              False             False      0.002175      Approve
  FREQ_C0561_1       C0561 3849.77  Kolkata   Amazon           False            False            False              False             False      0.002175      Approve
  FREQ_C0794_1       C0794 3261.07  Kolkata      Max           False            False            False              False             False      0.002175      Approv